<a href="https://colab.research.google.com/github/AndresAvella09/Proyecto_Multimodal/blob/main/notebooks/09_fusion_incertidumbre.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 09 — Fusión ponderada por incertidumbre

**Qué dejaron el 06 y el 08:**
- La fusión ingenua (concatenación) ya supera a la mejor rama, y la fusión tardía con pesos fijos no le
  gana (06).
- Las tres ramas están **igual de sobreconfiadas** (T ≈ 1.08): no hay una modalidad "más confiable" en
  general, así que repesar por la confianza global de cada modalidad no tiene base (06, D).
- Con películas recientes **el texto pierde ~4 puntos de AP y la imagen no cambia**; la dependencia de la
  imagen y Î(X_img; Y | X_txt) suben (08). La fusión ingenua usa los mismos pesos para todas las películas.

**Idea:** decidir el peso de cada modalidad **película por película y género por género**, según qué tan
segura está cada rama en ese caso. Para cada película *i* y género *c*:

$$w_{ic} = \operatorname{softmax}_m\big(\alpha_{cm} - \beta\,H_{icm}\big), \qquad
z_{ic} = b_c + \lambda \sum_m w_{icm}\, \tilde z_{icm}$$

- $\tilde z_{icm}$: logit de la fuente *m* dividido por su temperatura del 06 (calibrado).
- $H_{icm}$: entropía binaria de esa salida (0 = segura, ln 2 = no sabe).
- $\alpha_{cm}$: preferencia fija del género por cada fuente (= fusión tardía por género).
- $\beta$: **cuánto peso se le quita a una fuente cuando está insegura**. Es el parámetro de interés.
- $\lambda, b_c$: reescalan la mezcla (promediar logits los encoge) y ajustan la prevalencia.

Con $\beta = 0$ queda exactamente la fusión tardía por género. Así el modelo sin incertidumbre es un
**caso particular (control anidado)** y el aporte de la incertidumbre se mide comparando los dos.

| Método | Fuentes que combina | Usa incertidumbre | Parámetros |
|---|---|---|---|
| `ingenua` | — (fusión ingenua del 06) | no | — |
| `tardia` | imagen, texto | no ($\beta = 0$) | 2C + 1 |
| `compuerta` | imagen, texto | sí | 2C + 2 |
| `tardia3` | imagen, texto, fusión ingenua | no | 3C + 1 |
| `compuerta3` | imagen, texto, fusión ingenua | sí | 3C + 2 |

Todo se ajusta **solo en dev de MM-IMDB** (log-loss), con los modelos congelados del 06 y para cada una
de las 5 semillas. Después se aplica sin cambios a los cuatro grupos del 08. Corre en CPU en pocos minutos.

### Criterios fijados antes de ver los resultados

1. **No inferioridad en distribución:** en `test_mmimdb`, el IC 95% de AP(método) − AP(ingenua) queda
   por encima de −0.5 puntos.
2. **Mejora con películas nuevas:** en `recientes_tmdb`, el IC 95% de AP(método) − AP(ingenua) queda
   por encima de 0.
3. **La incertidumbre aporta:** $\beta \neq 0$ (IC sobre semillas sin el 0) y el log-loss en
   `recientes_tmdb` es menor que el del control anidado (`tardia` / `tardia3`).
   El signo de $\beta$ se interpreta, no se exige (ver sección 3).
4. **La ventaja crece con la época:** la ganancia sobre la ingenua es mayor en `recientes_tmdb` que en
   `control_tmdb` (diferencia en diferencias, IC sin el 0).

Se evalúan para `compuerta` y `compuerta3`. Un resultado negativo también se reporta: diría que la
incertidumbre de las ramas no anuncia cuándo se equivocan con películas nuevas.

In [ ]:
import os, sys, json, subprocess
EN_COLAB = 'google.colab' in sys.modules
if EN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

REPO = '/content/Proyecto_Multimodal'
if os.path.exists(REPO):
    subprocess.run(['git', '-C', REPO, 'pull', '-q'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', 'https://github.com/AndresAvella09/Proyecto_Multimodal.git', REPO], check=True)
sys.path.insert(0, REPO)

import time
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from src.datos import cargar, aplicar_estandarizacion
from src.modelos import cargar_artefactos, predecir_logits
from src.metricas import (sigmoide, f1_por_clase, elegir_umbrales, ap_por_clase_rapido, ic95,
                          bce_por_etiqueta, ece)
from src.fusion import entropia_binaria, ajustar_compuerta, combinar

DATA = '/content/drive/MyDrive/proyecto_multimodal/data'
EMB = '/content/drive/MyDrive/proyecto_multimodal/embeddings'
RES = '/content/drive/MyDrive/proyecto_multimodal/results'

SEED, N_BOOT, MIN_POSITIVOS = 42, 1000, 10
MARGEN_NO_INFERIORIDAD = 0.005            # criterio 1, en AP (0.5 puntos)
GRILLA_UMBRALES = np.round(np.arange(0.05, 0.96, 0.05), 2)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(SEED)

plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.alpha': 0.3, 'figure.dpi': 110})
GRUPOS = ['test_mmimdb', 'control_mmimdb', 'control_tmdb', 'recientes_tmdb']
COL_GRUPO = {'test_mmimdb': '#94a3b8', 'control_mmimdb': '#2563eb', 'control_tmdb': '#10b981', 'recientes_tmdb': '#ea580c'}
COL_MET = {'ingenua': '#10b981', 'tardia': '#a3a3a3', 'compuerta': '#7c3aed', 'tardia3': '#525252', 'compuerta3': '#db2777'}

ENTRADAS = {'tardia': ['imagen', 'texto'], 'compuerta': ['imagen', 'texto'],
            'tardia3': ['imagen', 'texto', 'fusion_ingenua'], 'compuerta3': ['imagen', 'texto', 'fusion_ingenua']}
INCERT = {'tardia': False, 'compuerta': True, 'tardia3': False, 'compuerta3': True}
METODOS = ['ingenua'] + list(ENTRADAS)
ANIDADO = {'compuerta': 'tardia', 'compuerta3': 'tardia3'}   # método con incertidumbre -> su control

## 1. Modelos del 06, dev de MM-IMDB y los cuatro grupos del 08

Igual que en el 08: embeddings de TMDB estandarizados con las estadísticas de train de MM-IMDB y
mismos géneros comparados. Se suma **dev de MM-IMDB**, donde se ajustan las compuertas.

In [ ]:
art = cargar_artefactos(f'{RES}/exp2_hoy_artefactos.pt', device)
assert art['etiquetas'] == 'hoy', "se usan etiquetas de IMDb hoy: correr el 06 con ETIQUETAS='hoy'"
MODELOS = art['modelos']
SEMILLAS = sorted({s for _, s in MODELOS})
G_ENT, G_EVAL = art['generos_entrenamiento'], art['generos_evaluacion']

X_mm, Y_mm, idx_mm, _, _ = cargar(DATA, EMB, etiquetas='hoy')
Xz_mm = aplicar_estandarizacion(X_mm, art['stats_estandarizacion'])

tm = pd.read_parquet(f'{DATA}/tmdb_meta.parquet')
X_tm = {'img': np.load(f'{EMB}/tmdb_clip_img.npy'), 'txt': np.load(f'{EMB}/tmdb_distilbert_txt.npy')}
assert all(len(v) == len(tm) for v in X_tm.values())
Xz_tm = aplicar_estandarizacion(X_tm, art['stats_estandarizacion'])
Y_tm = tm[[f'hoy_{g}' for g in G_ENT]].values.astype(np.float32)
i_ctrl = np.where(tm['grupo'] == 'control')[0]
i_rec = np.where(tm['grupo'] == 'reciente')[0]
filas_ctrl = tm['fila_hdf5'].values[i_ctrl]

conjuntos = {
    'dev_mmimdb':     ({m: Xz_mm[m][idx_mm['dev']] for m in Xz_mm}, Y_mm[idx_mm['dev']]),
    'test_mmimdb':    ({m: Xz_mm[m][idx_mm['test']] for m in Xz_mm}, Y_mm[idx_mm['test']]),
    'control_mmimdb': ({m: Xz_mm[m][filas_ctrl] for m in Xz_mm}, Y_mm[filas_ctrl]),
    'control_tmdb':   ({m: Xz_tm[m][i_ctrl] for m in Xz_tm}, Y_tm[i_ctrl]),
    'recientes_tmdb': ({m: Xz_tm[m][i_rec] for m in Xz_tm}, Y_tm[i_rec]),
}

# Mismos géneros que el 08
cols_eval = np.array([G_ENT.index(g) for g in G_EVAL])
pos = np.stack([conjuntos[g][1][:, cols_eval].sum(0) for g in ['control_mmimdb', 'recientes_tmdb']])
GENEROS = [g for g, k in zip(G_EVAL, (pos >= MIN_POSITIVOS).all(0)) if k]
COLS = np.array([G_ENT.index(g) for g in GENEROS])
YC = {c: y[:, COLS] for c, (_, y) in conjuntos.items()}

# Logits calibrados: cada modelo y semilla dividido por su temperatura (ajustada en dev en el 06)
Z = {}
for c, (x, _) in conjuntos.items():
    for (n, s), modelo in MODELOS.items():
        xs = [torch.from_numpy(x[m]).to(device) for m in art['modalidades'][n]]
        Z[(c, n, s)] = predecir_logits(modelo, xs)[:, COLS] / art['temperaturas'][f'{n}_s{s}']

print(f"{len(MODELOS)} modelos | semillas {SEMILLAS} | {len(GENEROS)} géneros")
for c in conjuntos:
    print(f"  {c:15s} n={len(YC[c]):5d}")

## 2. Antes de ajustar: ¿la entropía avisa cuándo una rama se equivoca?

Si un modelo está bien calibrado, su log-loss esperado **es igual a su entropía media**: la sorpresa que
"anuncia" coincide con la que tiene. Por eso

$$\text{exceso} = \overline{\text{log-loss}} - \overline{H}$$

mide **cuánta sorpresa no anunciada** tiene cada rama (en nats por celda película×género). Cerca de 0 =
la entropía es una buena señal de error. Positivo = se equivoca con más seguridad de la que muestra.

La compuerta solo puede bajarle el peso al texto en las películas recientes **si el texto anuncia su
incertidumbre** (sube $H$). Si lo que sube es el exceso, el texto se equivoca "con seguridad" y la
compuerta no lo va a ver.

In [ ]:
filas = []
for c in conjuntos:
    for n in ['imagen', 'texto', 'fusion_ingenua']:
        H = np.mean([entropia_binaria(Z[(c, n, s)]).mean() for s in SEMILLAS])
        L = np.mean([bce_por_etiqueta(YC[c], Z[(c, n, s)]).mean() for s in SEMILLAS])
        filas.append({'conjunto': c, 'modelo': n, 'entropia_media': H, 'logloss_medio': L, 'exceso': L - H})
sorpresa = pd.DataFrame(filas)
tab = sorpresa.pivot(index='conjunto', columns='modelo', values=['entropia_media', 'logloss_medio', 'exceso'])
tab.loc[list(conjuntos)].round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
x = np.arange(len(conjuntos))
COL_MOD = {'imagen': '#f59e0b', 'texto': '#2563eb', 'fusion_ingenua': '#10b981'}
for k, n in enumerate(['imagen', 'texto', 'fusion_ingenua']):
    d = sorpresa[sorpresa['modelo'] == n].set_index('conjunto').loc[list(conjuntos)]
    axes[0].plot(x, d['entropia_media'], 'o--', color=COL_MOD[n], alpha=0.6)
    axes[0].plot(x, d['logloss_medio'], 'o-', color=COL_MOD[n], label=n)
    axes[1].bar(x + (k - 1) * 0.27, d['exceso'], width=0.27, color=COL_MOD[n], label=n)
axes[0].set_title('log-loss (línea) vs entropía anunciada (punteada)', fontsize=10)
axes[1].set_title('exceso = log-loss − entropía (sorpresa no anunciada)', fontsize=10)
axes[1].axhline(0, color='black', lw=0.8)
for ax in axes:
    ax.set_xticks(x, [c.replace('_', '\n') for c in conjuntos], fontsize=8)
axes[0].legend(fontsize=8); axes[0].set_ylabel('nats por celda')
plt.tight_layout(); plt.show()

## 3. Ajuste de las compuertas en dev

Se ajustan por máxima verosimilitud (L-BFGS sobre la log-loss de dev) para cada semilla. Los umbrales de
F1 también se eligen en dev, para cada método.

In [ ]:
t0 = time.time()
COMPUERTAS = {}
for s in SEMILLAS:
    for met, ent in ENTRADAS.items():
        Zdev = np.stack([Z[('dev_mmimdb', n, s)] for n in ent], axis=-1)
        COMPUERTAS[(met, s)] = ajustar_compuerta(Zdev, YC['dev_mmimdb'], INCERT[met])

def salida(met, c, s):
    # Logits combinados (n, C) y pesos (n, C, M); la ingenua no tiene pesos
    if met == 'ingenua':
        return Z[(c, 'fusion_ingenua', s)], None
    return combinar(COMPUERTAS[(met, s)], np.stack([Z[(c, n, s)] for n in ENTRADAS[met]], axis=-1))

SAL = {(met, c, s): salida(met, c, s) for met in METODOS for c in conjuntos for s in SEMILLAS}
UMB = {(met, s): elegir_umbrales(YC['dev_mmimdb'], sigmoide(SAL[(met, 'dev_mmimdb', s)][0]), GRILLA_UMBRALES)
       for met in METODOS for s in SEMILLAS}
print(f"{len(COMPUERTAS)} compuertas ajustadas en {time.time() - t0:.0f} s")

### ¿Qué aprendió la compuerta?

- **β**: cómo cambia el peso de una rama según su entropía en esa película. Ojo con el signo: al
  sumar logits, una rama segura **ya pesa más** sin compuerta, porque su |logit| es grande (es lo que
  hace un producto de expertos). Entonces:
  - β > 0: la compuerta **refuerza** ese efecto (a la rama insegura le quita todavía más peso);
  - β < 0: lo **modera** (la suma de logits ya le daba demasiado peso a la rama segura);
  - β = 0: con la escala del logit alcanza.
- **λ**: cuánto se reescala la mezcla.
- **Peso base de la imagen** por género: el peso que recibe cuando las dos ramas están igual de seguras.

In [ ]:
param = pd.DataFrame([{'metodo': met, 'semilla': s, 'beta': float(m.beta), 'lambda': float(m.log_lambda.exp())}
                      for (met, s), m in COMPUERTAS.items()])
print("Media ± IC 95% sobre semillas:")
for met in ENTRADAS:
    d = param[param['metodo'] == met]
    b, l = ic95(d['beta']), ic95(d['lambda'])
    print(f"  {met:11s} beta = {b[0]:+.3f} ± {b[1]:.3f} | lambda = {l[0]:.3f} ± {l[1]:.3f}")

# Peso base de cada fuente por género (softmax de alfa), promedio de semillas
alfa = {met: np.mean([torch.softmax(COMPUERTAS[(met, s)].alfa.detach(), -1).numpy() for s in SEMILLAS], 0)
        for met in ENTRADAS}
peso_base = pd.DataFrame({f'{met}_{fuente}': alfa[met][:, k] for met in ['compuerta', 'compuerta3']
                          for k, fuente in enumerate(ENTRADAS[met])}, index=GENEROS)
peso_base.sort_values('compuerta_imagen', ascending=False).round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
# Curva de la compuerta de 2 fuentes: peso de la imagen según la diferencia de entropías
dH = np.linspace(-0.7, 0.7, 200)
for s in SEMILLAS:
    m = COMPUERTAS[('compuerta', s)]
    a = m.alfa.detach().numpy()
    a_med = np.median(a[:, 0] - a[:, 1])
    axes[0].plot(dH, 1 / (1 + np.exp(-(a_med + float(m.beta) * dH))), color=COL_MET['compuerta'], alpha=0.7)
axes[0].set_xlabel('H_texto − H_imagen (nats)  →  el texto está más inseguro')
axes[0].set_ylabel('peso de la imagen'); axes[0].set_ylim(0, 1)
axes[0].set_title('Compuerta (género mediano, una línea por semilla)', fontsize=10)

# Distribución de esa diferencia en control y recientes (semilla 0)
s0 = SEMILLAS[0]
for g in ['control_tmdb', 'recientes_tmdb']:
    d = (entropia_binaria(Z[(g, 'texto', s0)]) - entropia_binaria(Z[(g, 'imagen', s0)])).ravel()
    axes[1].hist(d, bins=60, range=(-0.7, 0.7), density=True, histtype='step', lw=1.8, color=COL_GRUPO[g],
                 label=f'{g} (media {d.mean():+.3f})')
axes[1].set_xlabel('H_texto − H_imagen (nats)'); axes[1].legend(fontsize=8)
axes[1].set_title('¿Las recientes llegan con el texto más inseguro?', fontsize=10)
plt.tight_layout(); plt.show()

## 4. Resultados por grupo (media ± IC 95% sobre semillas)

Mismas medidas para todos los métodos. `peso_imagen` es el peso medio que la compuerta le da a la
imagen en ese grupo: si la compuerta se adapta, debería subir en `recientes_tmdb`.

In [ ]:
filas = []
for c in GRUPOS:
    Y = YC[c]
    for met in METODOS:
        for s in SEMILLAS:
            z, w = SAL[(met, c, s)]
            P = (sigmoide(z) >= UMB[(met, s)]).astype(np.float32)
            filas.append({'grupo': c, 'metodo': met, 'semilla': s,
                          'ap_macro': np.nanmean(ap_por_clase_rapido(Y, z)),
                          'f1_macro': f1_por_clase(Y, P).mean(),
                          'logloss': bce_por_etiqueta(Y, z).sum(1).mean(),
                          'ece': ece(Y, sigmoide(z)),
                          'peso_imagen': np.nan if w is None else w[..., 0].mean()})
por_semilla = pd.DataFrame(filas)

fmt = lambda x: '{:.4f} ± {:.4f}'.format(*ic95(x)) if x.notna().all() else '—'
for medida in ['ap_macro', 'f1_macro', 'logloss', 'peso_imagen']:
    print(f"\n=== {medida} ===")
    print(por_semilla.pivot_table(index='metodo', columns='grupo', values=medida, aggfunc=fmt)
          .loc[METODOS, GRUPOS].to_string())

In [ ]:
# Diferencias pareadas por semilla contra la ingenua (puntos de AP)
filas = []
for c in GRUPOS:
    base = por_semilla[(por_semilla['grupo'] == c) & (por_semilla['metodo'] == 'ingenua')].set_index('semilla')
    for met in METODOS[1:]:
        d = por_semilla[(por_semilla['grupo'] == c) & (por_semilla['metodo'] == met)].set_index('semilla')
        m, h = ic95(100 * (d['ap_macro'] - base['ap_macro']))
        filas.append({'grupo': c, 'metodo': met, 'dif_ap': m, 'ic': h})
dif_sem = pd.DataFrame(filas)

fig, ax = plt.subplots(figsize=(10, 3.8))
x = np.arange(len(GRUPOS))
for k, met in enumerate(METODOS[1:]):
    d = dif_sem[dif_sem['metodo'] == met].set_index('grupo').loc[GRUPOS]
    ax.errorbar(x + (k - 1.5) * 0.1, d['dif_ap'], yerr=d['ic'], fmt='o', capsize=3, color=COL_MET[met], label=met)
ax.axhline(0, color='black', lw=0.8)
ax.axhline(-100 * MARGEN_NO_INFERIORIDAD, color='gray', lw=0.8, ls=':', label='margen de no inferioridad')
ax.set_xticks(x, [g.replace('_', '\n') for g in GRUPOS], fontsize=8)
ax.set_ylabel('AP macro − ingenua (puntos)'); ax.legend(fontsize=8, ncol=3)
ax.set_title('Cada método contra la fusión ingenua (media ± IC 95% sobre semillas)')
plt.tight_layout(); plt.show()

## 5. Bootstrap por películas y criterios

Igual que en el 08, la variabilidad que importa al comparar es la de **la muestra de películas**. Se
usan los logits promedio de las 5 semillas y se re-muestrean películas:

- Dentro de cada grupo las comparaciones son **pareadas** (mismas películas para los dos métodos).
- La diferencia en diferencias (`recientes_tmdb` vs `control_tmdb`) re-muestrea cada grupo por separado.

In [ ]:
ZP = {(met, c): np.mean([SAL[(met, c, s)][0] for s in SEMILLAS], axis=0) for met in METODOS for c in GRUPOS}
def ap_macro(Y, z): return np.nanmean(ap_por_clase_rapido(Y, z))
def logloss(Y, z): return bce_por_etiqueta(Y, z).sum(1).mean()
MEDIDAS = {'ap_macro': ap_macro, 'logloss': logloss}

def evaluar(c, r):
    return np.array([[f(YC[c][r], ZP[(met, c)][r]) for f in MEDIDAS.values()] for met in METODOS])

rng = np.random.default_rng(SEED)
t0 = time.time()
B = {c: [] for c in GRUPOS}                       # (N_BOOT, métodos, medidas)
for _ in range(N_BOOT):
    for c in GRUPOS:
        n = len(YC[c])
        B[c].append(evaluar(c, rng.integers(0, n, n)))
B = {c: np.array(v) for c, v in B.items()}
PUNTO = {c: evaluar(c, slice(None)) for c in GRUPOS}
print(f"Bootstrap: {time.time() - t0:.0f} s")

im = {m: k for k, m in enumerate(METODOS)}
filas = []
def agregar(comparacion, grupo, medida, j, punto, boot):
    lo, hi = np.percentile(boot, [2.5, 97.5])
    filas.append({'comparacion': comparacion, 'grupo': grupo, 'medida': medida, 'diferencia': punto,
                  'ic_inf': lo, 'ic_sup': hi})

for j, medida in enumerate(MEDIDAS):
    for c in GRUPOS:
        for met in METODOS[1:]:
            agregar(f'{met} − ingenua', c, medida, j, PUNTO[c][im[met], j] - PUNTO[c][0, j],
                    B[c][:, im[met], j] - B[c][:, 0, j])
        for met, ctrl in ANIDADO.items():
            agregar(f'{met} − {ctrl}', c, medida, j, PUNTO[c][im[met], j] - PUNTO[c][im[ctrl], j],
                    B[c][:, im[met], j] - B[c][:, im[ctrl], j])
    for met in METODOS[1:]:   # diferencia en diferencias: ganancia en recientes − ganancia en control
        g = lambda arr, c: arr[..., im[met], j] - arr[..., 0, j]
        agregar(f'{met} − ingenua (recientes − control)', 'dif_en_dif', medida, j,
                g(PUNTO['recientes_tmdb'], 0) - g(PUNTO['control_tmdb'], 0),
                g(B['recientes_tmdb'], 0) - g(B['control_tmdb'], 0))
boot = pd.DataFrame(filas)
boot['sin_cero'] = (boot['ic_inf'] > 0) | (boot['ic_sup'] < 0)

vista = boot[boot['medida'] == 'ap_macro'].copy()
vista['texto'] = vista.apply(lambda r: f"{100 * r.diferencia:+.2f} [{100 * r.ic_inf:+.2f}, {100 * r.ic_sup:+.2f}]"
                                       f"{' *' if r.sin_cero else ''}", axis=1)
print("AP macro, puntos (bootstrap 95%; * = sin el 0):")
vista.pivot(index='comparacion', columns='grupo', values='texto').fillna('')

In [ ]:
def fila(comp, grupo, medida='ap_macro'):
    return boot[(boot['comparacion'] == comp) & (boot['grupo'] == grupo) & (boot['medida'] == medida)].iloc[0]

criterios = []
for met, ctrl in ANIDADO.items():
    c1 = fila(f'{met} − ingenua', 'test_mmimdb')
    c2 = fila(f'{met} − ingenua', 'recientes_tmdb')
    b_m, b_h = ic95(param[param['metodo'] == met]['beta'])
    c3 = fila(f'{met} − {ctrl}', 'recientes_tmdb', 'logloss')
    c4 = fila(f'{met} − ingenua (recientes − control)', 'dif_en_dif')
    criterios += [
        {'metodo': met, 'criterio': '1. no inferior en test_mmimdb (AP)',
         'valor': f"IC inf {100 * c1.ic_inf:+.2f} pts (margen −{100 * MARGEN_NO_INFERIORIDAD:.1f})",
         'cumple': c1.ic_inf > -MARGEN_NO_INFERIORIDAD},
        {'metodo': met, 'criterio': '2. mejora en recientes_tmdb (AP)',
         'valor': f"{100 * c2.diferencia:+.2f} [{100 * c2.ic_inf:+.2f}, {100 * c2.ic_sup:+.2f}] pts",
         'cumple': c2.ic_inf > 0},
        {'metodo': met, 'criterio': '3a. beta distinto de 0',
         'valor': f"{b_m:+.3f} ± {b_h:.3f}", 'cumple': abs(b_m) > b_h},
        {'metodo': met, 'criterio': f'3b. log-loss menor que {ctrl} en recientes',
         'valor': f"{c3.diferencia:+.4f} [{c3.ic_inf:+.4f}, {c3.ic_sup:+.4f}] nats", 'cumple': c3.ic_sup < 0},
        {'metodo': met, 'criterio': '4. ganancia mayor en recientes que en control',
         'valor': f"{100 * c4.diferencia:+.2f} [{100 * c4.ic_inf:+.2f}, {100 * c4.ic_sup:+.2f}] pts",
         'cumple': c4.ic_inf > 0},
    ]
criterios = pd.DataFrame(criterios)
criterios

## 6. Diagnóstico: ¿falla la familia de compuertas o falla ajustarla en dev?

Si los criterios 2–4 no se cumplen hay dos explicaciones posibles:

- **La forma de la compuerta no sirve** para estas películas, aun con los mejores parámetros posibles.
- **La forma sirve, pero los parámetros de dev no transfieren**: con películas viejas de MM-IMDB la
  compuerta aprende a confiar en el texto, y eso deja de valer con las recientes.

Para separarlas se hace un **oráculo**: se ajusta la compuerta en la mitad A de las recientes y se evalúa
en la mitad B, contra la misma compuerta ajustada en dev. **No es un método válido** (usa etiquetas de
las recientes); es solo una cota de lo que la forma de compuerta podría lograr.

In [ ]:
rng_o = np.random.default_rng(SEED)
n_r = len(YC['recientes_tmdb'])
perm = rng_o.permutation(n_r)
A, Bm = perm[:n_r // 2], perm[n_r // 2:]
Yr = YC['recientes_tmdb']
filas = []
for s in SEMILLAS:
    filas.append({'metodo': 'ingenua', 'ajuste': '—', 'semilla': s,
                  'ap_B': ap_macro(Yr[Bm], Z[('recientes_tmdb', 'fusion_ingenua', s)][Bm])})
    for met in ['compuerta', 'compuerta3']:
        Zr = np.stack([Z[('recientes_tmdb', n, s)] for n in ENTRADAS[met]], axis=-1)
        oraculo = ajustar_compuerta(Zr[A], Yr[A], True)
        for ajuste, m in [('dev', COMPUERTAS[(met, s)]), ('oráculo (mitad A)', oraculo)]:
            filas.append({'metodo': met, 'ajuste': ajuste, 'semilla': s, 'beta': float(m.beta),
                          'ap_B': ap_macro(Yr[Bm], combinar(m, Zr[Bm])[0])})
oraculo_df = pd.DataFrame(filas)
oraculo_df.groupby(['metodo', 'ajuste']).agg(
    ap_B=('ap_B', lambda x: '{:.4f} ± {:.4f}'.format(*ic95(x))),
    beta=('beta', lambda x: '{:+.3f} ± {:.3f}'.format(*ic95(x)) if x.notna().all() else '—'))

**Cómo leer:** si el oráculo supera claramente a la ingenua y la versión de dev no, el problema es de
**transferencia** (el método necesitaría adaptarse a la población nueva, lo que motiva el dial del 10).
Si ni el oráculo mejora, la información para repesar película por película **no está en la entropía**
de las ramas: coincide con un exceso de sorpresa alto en la sección 2.

## Guardado

- `exp4_por_semilla.csv`: medidas por grupo, método y semilla.
- `exp4_bootstrap.csv`: comparaciones con IC bootstrap (AP y log-loss).
- `exp4_criterios.csv`: los cuatro criterios fijados de antemano.
- `exp4_parametros.csv`, `exp4_peso_base_por_genero.csv`: β, λ y pesos base de cada compuerta.
- `exp4_sorpresa.csv`, `exp4_oraculo.csv`: diagnósticos de las secciones 2 y 6.
- `exp4_compuertas.pt`: las compuertas ajustadas y sus umbrales (para el 10 y la app;
  se leen con `src.fusion.cargar_compuertas`).

In [ ]:
por_semilla.to_csv(f'{RES}/exp4_por_semilla.csv', index=False)
boot.to_csv(f'{RES}/exp4_bootstrap.csv', index=False)
criterios.to_csv(f'{RES}/exp4_criterios.csv', index=False)
param.to_csv(f'{RES}/exp4_parametros.csv', index=False)
peso_base.to_csv(f'{RES}/exp4_peso_base_por_genero.csv')
sorpresa.to_csv(f'{RES}/exp4_sorpresa.csv', index=False)
oraculo_df.to_csv(f'{RES}/exp4_oraculo.csv', index=False)
torch.save({'entradas': ENTRADAS, 'incertidumbre': INCERT, 'generos': GENEROS,
            'compuertas': {f'{met}_s{s}': m.state_dict() for (met, s), m in COMPUERTAS.items()},
            'umbrales': {f'{met}_s{s}': u for (met, s), u in UMB.items()}}, f'{RES}/exp4_compuertas.pt')
with open(f'{RES}/exp4_config.json', 'w') as fh:
    json.dump({'generos': GENEROS, 'semillas': SEMILLAS, 'n_bootstrap': N_BOOT, 'semilla_bootstrap': SEED,
               'margen_no_inferioridad': MARGEN_NO_INFERIORIDAD, 'metodos': METODOS,
               'entradas': ENTRADAS, 'n_por_conjunto': {c: int(len(YC[c])) for c in conjuntos}}, fh, indent=2)
print(sorted(f for f in os.listdir(RES) if f.startswith('exp4')))